## Pegasus Evaluation: Optimization

Section C.2. Setup, methods and metrics: `2a_base_case.ipynb`; parameters: `pegasus/evaluation_setup.py`. C.2 runs on
synthetic data, because the real data does not support it. C.1: `2e_finetuning.ipynb`.

In [1]:
# Data source of every experiment:
# - "synthetic": generated services and a simulated chain of queues (pegasus/data.py, pegasus/simulation.py)
# - "real": the measured runs of the second testbed campaign in statics/results_milos_v2 (pegasus/real_data.py)
data_source = "real"

# the parameters and helpers that every evaluation notebook shares, defined once in pegasus/evaluation_setup.py
from pegasus.evaluation_setup import *
import pandas as pd
from pegasus.figures import math_label, sweep_table, time_table
p_label = math_label(p_label)   # the quantile name in the math notation of the paper, e.g. $P_{95}$, for the figures and the text
# Held-out runs of the chain: up to held_out_workers workers per service, as B.1 on all chains in 2a_base_case.ipynb (the
# other notebooks stop at three, DEFAULT_CHAINS). The chain option is added to CHAIN_FOLDERS here instead of in
# pegasus/evaluation_setup.py, because a change there would recompute the cached results of every experiment.
held_out_workers = 4
c_chains = f"b1+b1u-max{held_out_workers}"
CHAIN_FOLDERS.setdefault(c_chains, (CAMPAIGN_V2, f"+b1u-max{held_out_workers}"))
globals().update(configure(data_source, c_chains))
# the base case of B.1 that this notebook builds on: read from statics/evaluation_cache/ (written by the first
# notebook that needs it), or computed here if the cache is missing or stale (b1_base)
globals().update(b1_base(data_source, chains=c_chains))
print(f"{data_source} data, {len(seeds)} seeds: chain {' -> '.join(base.path)}, per seed {n_train} training configurations per service and "
      f"{len(base.measured)} held-out configurations, {arrivals}")

base case of B.1 loaded from statics/evaluation_cache/b1_real+b1+b1u-max4_24ab358e8266af3d.joblib
real data, 3 seeds: chain detector -> transformer -> compressor, per seed 25 training configurations per service and 149 held-out configurations, poisson arrivals at one rate per configuration (1.58 to 65/s)


### C.2 Inferring Faster Service Compositions

> **TODO: reformulate C.2.** The objective below maximizes a weighted SLO fulfillment over quality and latency. What
> the experiment should answer instead: what is the *minimum budget of resources* (the cost of a configuration) at
> which the chain reaches the target latency with the target accuracy? Latency and accuracy then become constraints
> that a configuration must meet, and the cost is what the optimizer minimizes. The optimizers, the choice among the
> existing runs and the refitting of the models after every evaluated run can stay; the objective, the metrics and the
> conclusions below have to change.

An optimizer evaluates `c2_budget` existing runs of the chain of B.1 (synthetic data, one arrival rate for all runs),
starting from the same random run for every method, and refits the models of its method after every evaluated run
(`pegasus/optimization.py`). The objective is the weighted mean of the SLO fulfillments: the latency SLO
(`latency_target`, weight `latency_weight`) and the quality SLOs (`quality_slos`).

- **Optimizers:** *best of the runs* (largest predicted objective among the unevaluated runs, as Fluxion's random
  optimizer) and *gradient steps* (the unevaluated run closest to one step along the gradient of the predicted objective).
- **Methods:** Pegasus Node and the baselines of B.2 to B.6; reference: random search.
- **Promises:** a method *promises* the latency SLO of a chosen run if it predicts a latency within the target; the
  promise is *broken* if the measured quantile exceeds the target.

Figures: `figures/tsc/c2_optimization_<data source>_<optimizer>`.

In [ ]:
# C.2 needs every run at one common arrival rate, which the real data does not have (`skipped`): with real data it runs on
# synthetic data instead, on the synthetic base case of B.1 (b1_base), while C.1 above keeps the real data
c2_source = "synthetic" if "C.2" in skipped else data_source
c2_setup = {**configure(c2_source), **b1_base(c2_source)}   # the options and the base case of B.1 of that source

# C.2: optimizers that choose among the existing runs of the chain of B.1 with the latency of a method. This cell imports what
# it needs itself, so that nothing above has to be run again.
from pegasus.figures import METHOD_STYLES
from pegasus.optimization import (BEST_OF_RUNS, COLORS as C2_COLORS, GRADIENT, RANDOM_SEARCH, STYLES as C2_STYLES, ChainProblem, best_found,
                                  latency_promises, optimization_figure, optimize, real_observation, real_quality_slos, synthetic_observation)

METHOD_COLORS.update(C2_COLORS)
METHOD_STYLES.update(C2_STYLES)
c2_budget = 8          # runs of the chain that an optimizer evaluates, including the starting run
latency_weight = 0.5   # weight of the latency SLO in the objective; the quality SLOs share the rest equally
c2_names = [RANDOM_SEARCH] + [name for name in sweep_competitors if name != PROPOSED_SNC]   # Pegasus SNC only on the synthetic trees of B.3


def compute():
    if c2_source == "synthetic":
        c2_bases = c2_setup["bases"]        # B.1 already has one arrival rate for all configurations
        quality_slos = [(None, 1, 1.5)]     # the quality knob (column 1 of the configuration that all services share) should be at least 1.5
        observation = lambda s: synthetic_observation(c2_bases[s], s)
    else:
        c2_rate = 10.0                      # items/s for every run of the chain
        c2_bases = {s: c2_setup["chain_data"](s, arrival_rate=c2_rate) for s in seeds}
        # resolution (px) of the QR-code detector and of the object detector, and the larger model of the object detector and the face detector
        quality_slos = real_quality_slos(base_experiment, {("qr", "data_quality"): 800, ("cv", "data_quality"): 288, ("cv", "model_size"): 2,
                                                           ("face", "model_size"): 2})
        observation = lambda s: real_observation(base_experiment, c2_bases[s])
    c2_measured = c2_bases[seeds[0]].measured
    latency_target = float(np.median(c2_measured[np.isfinite(c2_measured)]))   # s: about half of the existing runs meet it

    c2_found = {optimizer: {name: np.empty((c2_budget, len(seeds))) for name in c2_names} for optimizer in (BEST_OF_RUNS, GRADIENT)}
    c2_promises = {optimizer: {name: dict(chosen=0, violated=0, promised=0, broken=0) for name in c2_names} for optimizer in c2_found}   # counts over all seeds
    c2_best_true = []   # per seed: the largest true SLO fulfillment of any existing run
    with warnings.catch_warnings():
        # the models are fitted anew after every evaluated run, and scikit-learn warns where their hyperparameters end at their bounds
        warnings.simplefilter("ignore", ConvergenceWarning)
        for j, s in enumerate(seeds):
            problem = ChainProblem(c2_bases[s], quality_slos, latency_target, latency_weight, observation(s), alpha, confidence, n_calibration, seed + s)
            c2_best_true.append(problem.true_fulfillment().max())
            start = int(np.random.default_rng([seed, s, 400]).integers(len(problem.z)))   # the same starting run for every method
            for optimizer in c2_found:
                for name in c2_names:
                    visited, promised = optimize(problem, None if name == RANDOM_SEARCH else name, optimizer, start, c2_budget,
                                                 np.random.default_rng([seed, s, 401]))
                    c2_found[optimizer][name][:, j] = best_found(problem, visited)
                    for key, count in latency_promises(problem, visited, promised).items():
                        c2_promises[optimizer][name][key] += count
    return dict(c2_bases=c2_bases, latency_target=latency_target, c2_found=c2_found, c2_promises=c2_promises, c2_best_true=c2_best_true,
                c2_runs=len(problem.z))


globals().update(cached("c2_optimization", c2_source, compute, inputs=dict(c2_budget=c2_budget, latency_weight=latency_weight, c2_names=c2_names)))
print(f"{len(c2_names) - 1} methods under {len(c2_found)} optimizers evaluated for {len(seeds)} seeds; {c2_runs} existing runs, "
      f"latency target {1e3 * latency_target:.0f} ms.")

In [ ]:
optimization_figure(c2_found, f"C.2 optimizing the configuration, {c2_source} data: {' → '.join(c2_setup['base'].path)}, "
                              f"{arrival_text(c2_bases[seeds[0]].workload)}, {len(seeds)} seeds", name=f"c2_optimization_{c2_source}")

In [ ]:
after = lambda optimizer, name, evaluations: np.median(c2_found[optimizer][name][evaluations - 1])   # median over the seeds
found_best = lambda optimizer, name: int(np.sum(np.isclose(c2_found[optimizer][name][-1], 1.0)))       # seeds in which the best existing run was found
print(f"{c2_source} data, latency target {1e3 * latency_target:.0f} ms; best existing run: true SLO fulfillment {np.median(c2_best_true):.2f} "
      "(median over the seeds)")
display(pd.DataFrame([{"optimizer": optimizer, "method": name, "best found after 2 (% of best run)": f"{after(optimizer, name, 2):.0%}",
                       f"after {c2_budget}": f"{after(optimizer, name, c2_budget):.0%}", "seeds that found the best run": f"{found_best(optimizer, name)} / {len(seeds)}",
                       "chosen runs": c["chosen"], "violate latency SLO": f"{c['violated']} ({c['violated'] / c['chosen']:.0%})",
                       "promised": c["promised"], "promises broken": f"{c['broken']} ({c['broken'] / c['promised']:.0%})" if c["promised"] else "–"}
                      for optimizer in c2_found for name, c in c2_promises[optimizer].items()]).set_index(["optimizer", "method"]))